In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter your Google API Key: ")

print("Setup complete.")

Setup complete.


In [2]:
import os
import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai

# Configure Gemini
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])
gemini = genai.GenerativeModel("gemini-3-flash-preview")

C:\Users\Meghana Veeramallu\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Meghana Veeramallu\AppData\Local\Temp\ipykernel_25972\3163585844.py:6: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [3]:
corpus = [
    "Self-attention computes relationships between all tokens in a sequence using query, key, and value matrices.",
    "Multi-head attention allows transformers to attend to information from different representation subspaces simultaneously.",
    "Positional encoding injects order information into transformer models that otherwise process tokens in parallel.",
    "Stochastic gradient descent updates model parameters using small random batches instead of the full dataset.",
    "Adam optimizer adapts learning rates for each parameter using first and second moment estimates.",
    "Learning rate scheduling dynamically adjusts the step size to stabilize and accelerate training.",
    "Overfitting happens when a model memorizes training data but fails to generalize to unseen data.",
    "Dropout randomly disables neurons during training to reduce co-adaptation and improve generalization.",
    "L2 regularization penalizes large weights to encourage simpler models.",
    "Tokenization converts raw text into tokens such as words, subwords, or characters for model processing.",
    "Named Entity Recognition identifies entities like people, locations, and organizations in text.",
    "BERT uses bidirectional context to understand the meaning of words based on both left and right context.",
    "The softmax function normalizes logits into a probability distribution across classes.",
    "Cross-entropy loss measures the divergence between predicted probabilities and true labels.",
    "Convolutional neural networks extract spatial features using filters applied over image grids."
]

print(f"Corpus size: {len(corpus)} documents")
for i, doc in enumerate(corpus):
    print(f"[{i:02d}] {doc[:80]}..." if len(doc) > 80 else f"[{i:02d}] {doc}")

Corpus size: 15 documents
[00] Self-attention computes relationships between all tokens in a sequence using que...
[01] Multi-head attention allows transformers to attend to information from different...
[02] Positional encoding injects order information into transformer models that other...
[03] Stochastic gradient descent updates model parameters using small random batches ...
[04] Adam optimizer adapts learning rates for each parameter using first and second m...
[05] Learning rate scheduling dynamically adjusts the step size to stabilize and acce...
[06] Overfitting happens when a model memorizes training data but fails to generalize...
[07] Dropout randomly disables neurons during training to reduce co-adaptation and im...
[08] L2 regularization penalizes large weights to encourage simpler models.
[09] Tokenization converts raw text into tokens such as words, subwords, or character...
[10] Named Entity Recognition identifies entities like people, locations, and organiz...
[11] BER

In [9]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import numpy as np

class HybridRetriever:
    
    def __init__(self, corpus: list[str], k: int = 60):
        
        self.corpus = corpus
        self.k = k

        # --- BM25 index ---
        tokenized_corpus = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized_corpus)

        # --- SBERT index ---
        print("Loading SBERT model...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        print("Encoding corpus with SBERT...")
        self.corpus_embeddings = self.sbert.encode(corpus, convert_to_numpy=True)
        print("HybridRetriever ready.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        
        n = len(self.corpus)

        # --- BM25 ranking ---
        bm25_scores = self.bm25.get_scores(query.lower().split())
        # argsort ascending, then reverse for descending rank
        bm25_order = np.argsort(bm25_scores)[::-1]  # indices sorted best→worst
        # Map doc_id → 1-based rank
        bm25_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}

        # --- SBERT ranking ---
        query_embedding = self.sbert.encode([query], convert_to_numpy=True)
        sbert_scores = cosine_similarity(query_embedding, self.corpus_embeddings)[0]
        sbert_order = np.argsort(sbert_scores)[::-1]
        sbert_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_order)}

        # --- RRF fusion ---
        # RRF(d) = 1/(k + r_BM25(d)) + 1/(k + r_SBERT(d))
        rrf_scores = {}
        for doc_id in range(n):
            rrf_scores[doc_id] = (
                1.0 / (self.k + bm25_ranks[doc_id])
                + 1.0 / (self.k + sbert_ranks[doc_id])
            )

        # Sort by RRF score descending and take top_k
        sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]

        results = [
            {
                "doc_id": doc_id,
                "rrf_score": round(score, 6),
                "bm25_rank": bm25_ranks[doc_id],
                "sbert_rank": sbert_ranks[doc_id],
                "text": self.corpus[doc_id],
            }
            for doc_id, score in sorted_docs
        ]
        return results

retriever = HybridRetriever(corpus)

Loading SBERT model...
Encoding corpus with SBERT...
HybridRetriever ready.


In [10]:
# Cross Encoder Re-ranker
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates, top_k=3):
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, c in enumerate(candidates):
        c["cross_score"] = scores[i]

    ranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return ranked[:top_k]

In [11]:
def hyde_expand(query: str) -> str:
    prompt = (
        "You are an expert in AI and Machine Learning. "
        "Write a concise, technically precise 1-3 sentence answer to the following question. "
        "Use domain-specific vocabulary as it would appear in a textbook or research paper. "
        "Do NOT say 'I think' or hedge — write the answer as a factual statement.\n\n"
        f"Question: {query}"
    )

    # temperature=0.0 → deterministic, factual hypothetical document
    response = gemini.generate_content(
        prompt,
        generation_config=genai.types.GenerationConfig(temperature=0.0)
    )
    return response.text.strip()


In [12]:
def advanced_rag(user_query: str, verbose: bool = False) -> str:
    
    # --- Step 1: HyDE Query Expansion ---
    hyde_document = hyde_expand(user_query)
    if verbose:
        print("[Step 1] HyDE Hypothesis:")
        print(f"  {hyde_document}\n")

    # --- Step 2: Hybrid Retrieval (query = HyDE doc) ---
    candidates = retriever.retrieve(hyde_document, top_k=5)
    if verbose:
        print("[Step 2] Hybrid Retrieval top-5 (RRF):")
        for c in candidates:
            print(f"  rrf={c['rrf_score']} bm25_rank={c['bm25_rank']} sbert_rank={c['sbert_rank']}")
            print(f"  {c['text'][:80]}")
        print()

    # --- Step 3: Cross-Encoder Re-Ranking (query = ORIGINAL user query) ---
    reranked = rerank(user_query, candidates, top_k=3)
    if verbose:
        print("[Step 3] Re-Ranked top-3 (Cross-Encoder):")
        for r in reranked:
            print(f"  ce_score={r['cross_encoder_score']:7.4f} | {r['text'][:80]}")
        print()

    # --- Step 4: LLM Generation ---
    context = "\n".join([f"- {r['text']}" for r in reranked])
    generation_prompt = (
        "You are a university AI/ML teaching assistant. "
        "Answer the student's question using ONLY the context provided below. "
        "Be concise, accurate, and educational.\n\n"
        f"Context:\n{context}\n\n"
        f"Student Question: {user_query}\n\n"
        "Answer:"
    )

    final_response = gemini.generate_content(
        generation_prompt,
        generation_config=genai.types.GenerationConfig(temperature=0.3)
    )
    answer = final_response.text.strip()

    if verbose:
        print("[Step 4] Final Answer:")
        print(f"  {answer}")

    return answer


In [13]:
# Pre-compute corpus embeddings for naïve RAG (reuses the SBERT model already loaded)
naive_embeddings = retriever.corpus_embeddings  # shape: (n_docs, embed_dim)


def naive_rag(user_query: str, top_k: int = 3, verbose: bool = False) -> str:

    # Dense retrieval only
    q_emb = retriever.sbert.encode([user_query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, naive_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    top_docs = [{"doc_id": int(i), "cosine_score": round(float(scores[i]), 4), "text": corpus[i]}
                for i in top_indices]

    if verbose:
        print("[Naïve RAG] Top documents (SBERT cosine):")
        for d in top_docs:
            print(f"  cosine={d['cosine_score']} | {d['text'][:80]}")
        print()

    context = "\n".join([f"- {d['text']}" for d in top_docs])
    generation_prompt = (
        "You are a university AI/ML teaching assistant. "
        "Answer the student's question using ONLY the context provided below. "
        "Be concise, accurate, and educational.\n\n"
        f"Context:\n{context}\n\n"
        f"Student Question: {user_query}\n\n"
        "Answer:"
    )

    response = gemini.generate_content(
        generation_prompt,
        generation_config=genai.types.GenerationConfig(temperature=0.3)
    )
    answer = response.text.strip()

    if verbose:
        print("[Naïve RAG] Answer:")
        print(f"  {answer}")

    return answer, top_docs[0]["text"]  # return answer + top doc for comparison table


print("Naïve RAG baseline ready.")

Naïve RAG baseline ready.


In [14]:
# Test Queries
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is overfitting?"
]

for q in queries:
    print("\nQUERY:", q)
    print("Naive:", naive_rag(q))
    print("Advanced:", advanced_rag(q))


QUERY: how do transformers encode meaning?
Naive: ('Based on the provided context, transformers process information through these key mechanisms:\n\n*   **Tokenization:** Raw text is converted into tokens (words, subwords, or characters) for processing.\n*   **Positional Encoding:** This injects order information into the model, as transformers otherwise process tokens in parallel.\n*   **Multi-head Attention:** This allows the model to attend to information from different representation subspaces simultaneously.', 'Positional encoding injects order information into transformer models that otherwise process tokens in parallel.')
Advanced: Transformers encode meaning through the following processes:

*   **Tokenization**: Raw text is first converted into tokens, such as words, subwords, or characters, to be processed by the model.
*   **Positional Encoding**: Because transformers process tokens in parallel, this step injects order information into the model.
*   **Multi-head Attention*

In [15]:
import time

results_table = []

for query in queries:
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print("="*70)

    # --- Naïve RAG (NO LLM call — just retrieval for comparison) ---
    q_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    scores = cosine_similarity(q_emb, naive_embeddings)[0]
    top_idx = np.argsort(scores)[::-1][0]
    naive_top_doc = corpus[top_idx]
    print(f"[Naïve RAG] Top doc: {naive_top_doc[:80]}")

    # --- Advanced RAG (1 HyDE call + 1 generation call = 2 Gemini calls) ---
    hyde_q = hyde_expand(query)
    adv_candidates = retriever.retrieve(hyde_q, top_k=5)
    adv_reranked = rerank(query, adv_candidates, top_k=3)
    adv_top_doc = adv_reranked[0]["text"]

    context = "\n".join([f"- {r['text']}" for r in adv_reranked])
    adv_response = gemini.generate_content(
        f"Answer this student question using only the context below.\n\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:",
        generation_config=genai.types.GenerationConfig(temperature=0.3)
    )
    adv_answer = adv_response.text.strip()
    print(f"[Advanced RAG] Top doc: {adv_top_doc[:80]}")
    print(f"[Advanced RAG] Answer: {adv_answer}")

    results_table.append({
        "query": query,
        "naive_top_doc": naive_top_doc,
        "advanced_top_doc": adv_top_doc,
        "different": naive_top_doc != adv_top_doc,
    })

    print(f"\n✓ Sleeping 15s...")
    time.sleep(15)

print("\nAll queries processed.")


QUERY: how do transformers encode meaning?
[Naïve RAG] Top doc: Positional encoding injects order information into transformer models that other
[Advanced RAG] Top doc: Multi-head attention allows transformers to attend to information from different
[Advanced RAG] Answer: Transformers encode meaning by using tokenization to convert raw text into tokens, injecting order information through positional encoding, and utilizing multi-head attention to attend to information from different representation subspaces simultaneously.

✓ Sleeping 15s...

QUERY: optimization techniques for training
[Naïve RAG] Top doc: Learning rate scheduling dynamically adjusts the step size to stabilize and acce
[Advanced RAG] Top doc: Learning rate scheduling dynamically adjusts the step size to stabilize and acce
[Advanced RAG] Answer: Optimization techniques for training include:
- **Learning rate scheduling:** Dynamically adjusts the step size to stabilize and accelerate training.
- **Dropout:** Randomly di

In [16]:
import pandas as pd

df = pd.DataFrame([
    {
        "Query": row["query"],
        "Naïve RAG Top Doc": row["naive_top_doc"][:120] + "...",
        "Advanced RAG Top Doc": row["advanced_top_doc"][:120] + "...",
        "Different?": "YES ✓" if row["different"] else "NO (same)"
    }
    for row in results_table
])

df.style.set_properties(**{
    'text-align': 'left',
    'white-space': 'pre-wrap',
    'max-width': '300px'
}).set_table_styles([
    {'selector': 'th', 'props': [('text-align', 'left'), ('background-color', '#f2f2f2')]}
])

,Query,Naïve RAG Top Doc,Advanced RAG Top Doc,Different?
0,how do transformers encode meaning?,Positional encoding injects order information into transformer models that otherwise process tokens in parallel....,Multi-head attention allows transformers to attend to information from different representation subspaces simultaneously...,YES ✓
1,optimization techniques for training,Learning rate scheduling dynamically adjusts the step size to stabilize and accelerate training....,Learning rate scheduling dynamically adjusts the step size to stabilize and accelerate training....,NO (same)
2,what is overfitting?,Overfitting happens when a model memorizes training data but fails to generalize to unseen data....,Overfitting happens when a model memorizes training data but fails to generalize to unseen data....,NO (same)


In [17]:
def weighted_rrf_retrieve(query: str, retriever: HybridRetriever, alpha: float, top_k: int = 3) -> list[dict]:
    
    n = len(retriever.corpus)

    # BM25 ranks
    bm25_scores = retriever.bm25.get_scores(query.lower().split())
    bm25_order = np.argsort(bm25_scores)[::-1]
    bm25_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_order)}

    # SBERT ranks
    q_emb = retriever.sbert.encode([query], convert_to_numpy=True)
    sbert_scores = cosine_similarity(q_emb, retriever.corpus_embeddings)[0]
    sbert_order = np.argsort(sbert_scores)[::-1]
    sbert_ranks = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_order)}

    k = retriever.k
    rrf_scores = {
        doc_id: alpha / (k + bm25_ranks[doc_id]) + (1 - alpha) / (k + sbert_ranks[doc_id])
        for doc_id in range(n)
    }

    sorted_docs = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [{"doc_id": doc_id, "rrf_score": round(score, 6), "text": retriever.corpus[doc_id]}
            for doc_id, score in sorted_docs]


# Compare α on keyword-heavy vs semantic queries
keyword_query = "BM25 probabilistic ranking function"
semantic_query = "how do neural networks understand language meaning"
alphas = [0.3, 0.5, 0.7]

for q_label, q in [("Keyword-heavy", keyword_query), ("Semantic", semantic_query)]:
    print(f"\n{'='*60}")
    print(f"Query type: {q_label}")
    print(f"Query: '{q}'")
    print("="*60)
    for alpha in alphas:
        top = weighted_rrf_retrieve(q, retriever, alpha=alpha, top_k=1)[0]
        print(f"  α={alpha} → [{top['doc_id']:02d}] {top['text'][:90]}")


Query type: Keyword-heavy
Query: 'BM25 probabilistic ranking function'
  α=0.3 → [12] The softmax function normalizes logits into a probability distribution across classes.
  α=0.5 → [12] The softmax function normalizes logits into a probability distribution across classes.
  α=0.7 → [12] The softmax function normalizes logits into a probability distribution across classes.

Query type: Semantic
Query: 'how do neural networks understand language meaning'
  α=0.3 → [11] BERT uses bidirectional context to understand the meaning of words based on both left and 
  α=0.5 → [11] BERT uses bidirectional context to understand the meaning of words based on both left and 
  α=0.7 → [11] BERT uses bidirectional context to understand the meaning of words based on both left and 
